# KPI Calculation
## Sugar Alternative Whitespace Analysis

**Objective:** Calculate 4 KPIs for each category to identify market opportunities


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## Part 1: Loading Data

In [ ]:
from google.colab import files

# Uploading processed product data
print("Upload us_products_processed.csv:")
uploaded = files.upload()

Upload us_products_processed.csv:


Saving us_products_processed.csv to us_products_processed.csv


In [ ]:
# Loading data
df = pd.read_csv('us_products_processed.csv', low_memory=False)

print(f"Loaded {len(df):,} products")
print(f"Categories: {df['branded_food_category'].nunique()}")
df['branded_food_category'].unique()

Loaded 49,478 products
Categories: 151


array(['Alcohol', 'All Noodles', 'Bacon, Sausages & Ribs',
       'Baking Additives & Extracts',
       'Baking Decorations & Dessert Toppings',
       'Baking/Cooking Mixes/Supplies', 'Biscuits/Cookies',
       'Biscuits/Cookies (Shelf Stable)', 'Bread', 'Bread & Muffin Mixes',
       'Breads & Buns', 'Breakfast Sandwiches, Biscuits & Meals',
       'Butter & Spread', 'Cake, Cookie & Cupcake Mixes',
       'Cakes, Cupcakes, Snack Cakes', 'Candy', 'Canned & Bottled Beans',
       'Canned Condensed Soup', 'Canned Fruit', 'Canned Meat',
       'Canned Seafood', 'Canned Soup', 'Canned Tuna',
       'Canned Vegetables', 'Cereal', 'Cheese',
       'Cheese/Cheese Substitutes', 'Chewing Gum & Mints', 'Chili & Stew',
       'Chips, Pretzels & Snacks', 'Chocolate', 'Coffee',
       'Confectionery Products', 'Cooked & Prepared',
       'Cookies & Biscuits', 'Crackers & Biscotti', 'Cream',
       'Croissants, Sweet Rolls, Muffins & Other Pastries',
       'Crusts & Dough', 'Deli Salads',
       '

## Part 2: Filtering Categories

In [ ]:
print("Filtering categories for reformulation potential:\n")

# Categories to exclude from analysis
exclude_categories = [
    # Pure sugar products
    'Granulated, Brown & Powdered Sugar',
    'Honey',
    'Syrups & Molasses',
    'Sugar Substitutes',

    # Plain ingredients (no sugar or very low)
    'Flours & Corn Meal',
    'Grains/Flour',
    'Herbs & Spices',
    'Herbs/Spices/Extracts',
    'Vegetable & Cooking Oils',
    'Oils Edible',
    'Water',
    'Plant Based Water',
    'Tea Bags',
    'Cream',

    # Unprocessed proteins (no sugar)
    'Fish  Unprepared/Unprocessed',
    'Shellfish Unprepared/Unprocessed',
    'Meat/Poultry/Other Animals  Unprepared/Unprocessed',
    'Eggs & Egg Substitutes',

    # Plain vegetables/grains (low sugar)
    'Frozen Vegetables',
    'Tomatoes',
    'Rice',
    'Other Grains & Seeds',

    # Supplements
    'Meal Replacement Supplements',
    'Specialty Formula Supplements'
]

# Filter 1: Removing excluded categories
initial_count = len(df)
df_filtered = df[~df['branded_food_category'].isin(exclude_categories)]
removed_excluded = initial_count - len(df_filtered)

print(f"Removed {removed_excluded:,} products from {len(exclude_categories)} excluded categories")

# Filter 2: Keep only categories with meaningful sugar (>5g average)
category_avg_sugar = df_filtered.groupby('branded_food_category')['sugar_100g'].mean()
meaningful_categories = category_avg_sugar[category_avg_sugar > 5].index

df_filtered = df_filtered[df_filtered['branded_food_category'].isin(meaningful_categories)]
removed_low_sugar = len(df) - removed_excluded - len(df_filtered)

print(f"Removed {removed_low_sugar:,} products from low-sugar categories (<5g average)")

# Final filtered dataset
print(f"\nFinal dataset:")
print(f"  Products: {len(df_filtered):,}")
print(f"  Categories: {df_filtered['branded_food_category'].nunique()}")

# Using filtered data for all subsequent analysis
df = df_filtered

Filtering categories for reformulation potential:

Removed 3,361 products from 24 excluded categories
Removed 16,148 products from low-sugar categories (<5g average)

Final dataset:
  Products: 29,969
  Categories: 65


## Part 3: KPI 1 - Sugar Load

In [ ]:
# Calculating Sugar Load
# Formula: average sugar per 100g / WHO daily limit (50g)

print("Calculating Sugar Load:")
WHO_LIMIT = 50  # World Health Organization (WHO) recommends max 50g sugar per day

# Calculating average sugar per category
sugar_by_category = df.groupby('branded_food_category')['sugar_100g'].mean()

# Calculating sugar load (proportion of daily limit)
sugar_load_by_category = sugar_by_category / WHO_LIMIT

print(f"Calculated for {len(sugar_load_by_category)} categories")
print("\nTop 10 categories by sugar load:")
print(sugar_load_by_category.sort_values(ascending=False).head(10))

Calculating Sugar Load:
Calculated for 65 categories

Top 10 categories by sugar load:
branded_food_category
Candy                                    1.195769
Baking Decorations & Dessert Toppings    1.114540
Jam, Jelly & Fruit Spreads               0.983711
Chocolate                                0.895654
Gelatin, Gels, Pectins & Desserts        0.887822
Confectionery Products                   0.869147
Fruit  Prepared/Processed                0.867521
Powdered Drinks                          0.858129
Wholesome Snacks                         0.830719
Cake, Cookie & Cupcake Mixes             0.697522
Name: sugar_100g, dtype: float64


## Part 4: KPI 2 - Market Gap

In [ ]:
# Calculating Market Gap
# Formula: 100 - (products with alternatives / total products * 100)

print("Calculating Market Gap:")

# Counting products with/without alternatives per category
gap_by_category = df.groupby('branded_food_category')['has_sugar_alternative'].agg(['sum', 'count'])
gap_by_category['market_gap_pct'] = 100 - (gap_by_category['sum'] / gap_by_category['count'] * 100)

print(f"Calculated for {len(gap_by_category)} categories")
print("\nTop 10 categories by market gap:")
print(gap_by_category.sort_values('market_gap_pct', ascending=False)['market_gap_pct'].head(10))

Calculating Market Gap:
Calculated for 65 categories

Top 10 categories by market gap:
branded_food_category
Baking Additives & Extracts                        100.000000
Fruit  Prepared/Processed                          100.000000
Gravy Mix                                          100.000000
Other Condiments                                   100.000000
Crackers & Biscotti                                 99.784017
Biscuits/Cookies                                    99.272727
Seasoning Mixes, Salts, Marinades & Tenderizers     99.246704
Bread & Muffin Mixes                                99.218750
Frozen Pancakes, Waffles, French Toast & Crepes     99.166667
Pickles, Olives, Peppers & Relishes                 99.129489
Name: market_gap_pct, dtype: float64


## Part 5: KPI 3 - Market Size

In [ ]:
# Calculating Market Size
# Formula: count of products per category

print("Calculating market size:\n")

# Counting products per category
size_by_category = df.groupby('branded_food_category').size()

print(f"Calculated for {len(size_by_category)} categories")
print("\nTop 10 largest categories:")
print(size_by_category.sort_values(ascending=False).head(10))

Calculating market size:

Calculated for 65 categories

Top 10 largest categories:
branded_food_category
Popcorn, Peanuts, Seeds & Related Snacks           2439
Candy                                              2378
Ice Cream & Frozen Yogurt                          1615
Cookies & Biscuits                                 1433
Breads & Buns                                      1246
Chocolate                                          1114
Fruit & Vegetable Juice, Nectars & Fruit Drinks    1085
Snack, Energy & Granola Bars                       1046
Cakes, Cupcakes, Snack Cakes                       1014
Yogurt                                              979
dtype: int64


## Part 6: Creating KPI Table and Calculating Opportunity Score

In [ ]:
# Formula: (sugar_load_norm * 0.35) + (market_gap * 0.40) + (market_size_norm * 0.25)

print("Creating KPI table:\n")

# Combining all metrics into one dataframe
kpi_table = pd.DataFrame({
    'category': sugar_by_category.index,
    'avg_sugar': sugar_by_category.values,
    'sugar_load': sugar_load_by_category.values,
    'market_gap': gap_by_category['market_gap_pct'].values,
    'market_size': size_by_category.values
})

# Normalizing KPIs to 0-100 scale
kpi_table['sugar_load_norm'] = (kpi_table['sugar_load'] / kpi_table['sugar_load'].max()) * 100
kpi_table['gap_norm'] = kpi_table['market_gap']  # Already 0-100
kpi_table['size_norm'] = (kpi_table['market_size'] / kpi_table['market_size'].max()) * 100

# Calculating initial opportunity score
kpi_table['opportunity_score'] = (
    kpi_table['sugar_load_norm'] * 0.35 +
    kpi_table['gap_norm'] * 0.40 +
    kpi_table['size_norm'] * 0.25
)

# Sorting by opportunity score
kpi_table = kpi_table.sort_values('opportunity_score', ascending=False).reset_index(drop=True)

print(f"KPI table created: {len(kpi_table)} categories")
print("\nKPI weights:")
print("  Sugar Load: 35%")
print("  Market Gap: 40%")
print("  Market Size: 25%")

print("\nTop 20 opportunities by combined score:")
kpi_table[['category', 'avg_sugar', 'market_gap', 'market_size', 'opportunity_score']].head(20)

Creating KPI table:

KPI table created: 65 categories

KPI weights:
  Sugar Load: 35%
  Market Gap: 40%
  Market Size: 25%

Top 20 opportunities by combined score:


,category,avg_sugar,market_gap,market_size,opportunity_score
0,Candy,59.788431,89.865433,2378,95.320917
1,Baking Decorations & Dessert Toppings,55.726982,98.017621,454,76.483033
2,"Popcorn, Peanuts, Seeds & Related Snacks",18.504362,98.400984,2439,75.192802
3,Chocolate,44.782675,86.714542,1114,72.320098
4,"Jam, Jelly & Fruit Spreads",49.185546,98.319328,357,71.780115
5,Cookies & Biscuits,33.007446,94.207955,1433,71.694056
6,Wholesome Snacks,41.535939,98.756906,724,71.238873
7,Fruit Prepared/Processed,43.376053,100.000000,38,65.781738
8,Ice Cream & Frozen Yogurt,21.210167,89.597523,1615,64.809304
9,"Cakes, Cupcakes, Snack Cakes",32.368067,87.278107,1014,64.253033


## Part 7: Identifying Top Categories for Google Trends

In [ ]:
print("Identifying top 10 categories for Google Trends collection:\n")

# Getting top 10 categories
top_10 = kpi_table.head(10)

print("Top 10 categories by opportunity score:")
for idx, row in top_10.iterrows():
    print(f"\n{idx+1}. {row['category']}")
    print(f"   Opportunity Score: {row['opportunity_score']:.1f}")
    print(f"   Sugar Load: {row['sugar_load']:.2f} ({row['avg_sugar']:.1f}g per 100g)")
    print(f"   Market Gap: {row['market_gap']:.1f}%")
    print(f"   Market Size: {row['market_size']:.0f} products")

# Saving top 10 list for next step
top_10_list = top_10['category'].tolist()
print(top_10_list)
with open('top_categories_for_trends.txt', 'w') as f:
    for cat in top_10_list:
        f.write(f"{cat}\n")

print("\nSaved: top_categories_for_trends.txt")
files.download('top_categories_for_trends.txt')

Identifying top 10 categories for Google Trends collection:

Top 10 categories by opportunity score:

1. Candy
   Opportunity Score: 95.3
   Sugar Load: 1.20 (59.8g per 100g)
   Market Gap: 89.9%
   Market Size: 2378 products

2. Baking Decorations & Dessert Toppings
   Opportunity Score: 76.5
   Sugar Load: 1.11 (55.7g per 100g)
   Market Gap: 98.0%
   Market Size: 454 products

3. Popcorn, Peanuts, Seeds & Related Snacks
   Opportunity Score: 75.2
   Sugar Load: 0.37 (18.5g per 100g)
   Market Gap: 98.4%
   Market Size: 2439 products

4. Chocolate
   Opportunity Score: 72.3
   Sugar Load: 0.90 (44.8g per 100g)
   Market Gap: 86.7%
   Market Size: 1114 products

5. Jam, Jelly & Fruit Spreads
   Opportunity Score: 71.8
   Sugar Load: 0.98 (49.2g per 100g)
   Market Gap: 98.3%
   Market Size: 357 products

6. Cookies & Biscuits
   Opportunity Score: 71.7
   Sugar Load: 0.66 (33.0g per 100g)
   Market Gap: 94.2%
   Market Size: 1433 products

7. Wholesome Snacks
   Opportunity Score: 71.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Part 8: Summary Statistics

In [ ]:
print("KPI summary statistics:\n")
print(kpi_table[['sugar_load', 'market_gap', 'market_size', 'opportunity_score']].describe())

# Finding categories with high scores on multiple KPIs
print("\nCategories scoring high on multiple KPIs:\n")

high_sugar = kpi_table['sugar_load'] > kpi_table['sugar_load'].quantile(0.75)
high_gap = kpi_table['market_gap'] > kpi_table['market_gap'].quantile(0.75)
large_market = kpi_table['market_size'] > kpi_table['market_size'].quantile(0.75)

multi_high = kpi_table[high_sugar & high_gap]

print(f"Categories with high sugar load AND high market gap: {len(multi_high)}")
print("\nTop opportunities:")
print(multi_high[['category', 'sugar_load', 'market_gap', 'market_size', 'opportunity_score']].head(10))

KPI summary statistics:

       sugar_load  market_gap  market_size  opportunity_score
count   65.000000   65.000000    65.000000          65.000000
mean     0.404000   87.305842   461.061538          51.473303
std      0.271863   18.812087   518.580963          12.843120
min      0.102362   11.111111    29.000000          13.503065
25%      0.172970   86.714542   119.000000          45.814645
50%      0.329143   95.076923   254.000000          50.371647
75%      0.524313   98.319328   653.000000          54.356819
max      1.195769  100.000000  2439.000000          95.320917

Categories scoring high on multiple KPIs:

Categories with high sugar load AND high market gap: 3

Top opportunities:
                     category  sugar_load  market_gap  market_size  \
6            Wholesome Snacks    0.830719   98.756906          724   
7   Fruit  Prepared/Processed    0.867521  100.000000           38   
12           Other Condiments    0.598714  100.000000           65   

    opportunity_s

## Part 9: Saving Results

In [ ]:
# Saving KPI table
kpi_table.to_csv('initial_kpi_results.csv', index=False)

print("Saved: initial_kpi_results.csv")
print(f"\nFile contains {len(kpi_table)} categories")
print(f"Columns: {list(kpi_table.columns)}")

files.download('initial_kpi_results.csv')

Saved: initial_kpi_results.csv

File contains 65 categories
Columns: ['category', 'avg_sugar', 'sugar_load', 'market_gap', 'market_size', 'sugar_load_norm', 'gap_norm', 'size_norm', 'opportunity_score']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
## Summary

print("\nAnalysis summary:\n")
print(f"Categories analyzed: {len(kpi_table)}")
print(f"Products analyzed: {len(df):,}")

print("\n3 KPIs calculated:")
print("  1. Sugar Load (avg sugar vs WHO daily limit)")
print("  2. Market Gap (% products without alternatives)")
print("  3. Market Size (number of products)")

print("\nTop 3 opportunities by combined score:")
for i in range(min(3, len(kpi_table))):
    row = kpi_table.iloc[i]
    print(f"\n{i+1}. {row['category']}")
    print(f"   Opportunity Score: {row['opportunity_score']:.1f}/100")
    print(f"   Sugar Load: {row['sugar_load']:.2f}")
    print(f"   Market Gap: {row['market_gap']:.1f}%")
    print(f"   Market Size: {row['market_size']:.0f} products")

print("\nNext step: Collect Google Trends data for top 10 categories")


Analysis summary:

Categories analyzed: 65
Products analyzed: 29,969

3 KPIs calculated:
  1. Sugar Load (avg sugar vs WHO daily limit)
  2. Market Gap (% products without alternatives)
  3. Market Size (number of products)

Top 3 opportunities by combined score:

1. Candy
   Opportunity Score: 95.3/100
   Sugar Load: 1.20
   Market Gap: 89.9%
   Market Size: 2378 products

2. Baking Decorations & Dessert Toppings
   Opportunity Score: 76.5/100
   Sugar Load: 1.11
   Market Gap: 98.0%
   Market Size: 454 products

3. Popcorn, Peanuts, Seeds & Related Snacks
   Opportunity Score: 75.2/100
   Sugar Load: 0.37
   Market Gap: 98.4%
   Market Size: 2439 products

Next step: Collect Google Trends data for top 10 categories
